In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV

from src.common.load_data import load_data, drop_targets, get_test_sets
from src.common.treat_missing import load_and_process_for_random_forest
from src.common.evaluations import evaluate_regression

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200

**How to fix overfitting in Random Forest?**

| If your model is overfitting | Action to take on Hyperparameter |
|---|---|
| max_depth | Decrease it (e.g., from None to 10)|
| min_samples_leaf | Increase it (e.g., from 1 to 5) |
| min_samples_split | Increase it (e.g., from 2 to 10) |
| max_features | Decrease it (e.g., from 1.0 to 'sqrt') | 
| ccp_alpha | Increase it (e.g., from 0.0 to 0.01) |

## 1. Load Data

In [5]:
df = load_data("./data/cibil_score/cibil_score.csv")
X, y_lin_reg, y_binary, y_multiclass = drop_targets(df)
X_train, X_test, y_train, y_test = get_test_sets(X, y_lin_reg, test_size=0.15)

0


## 2. Preprocessing

In [6]:
X_train_processed, X_test_processed = load_and_process_for_random_forest(X_train, X_test)

2026-09-01 17:06:29,285 [INFO] src.common.treat_missing: No columns exceeded the 95% null threshold; none dropped.
2026-09-01 17:06:29,363 [INFO] src.common.treat_missing: Fitted OrdinalEncoder on 5 categorical column(s): ['maritalstatus', 'education', 'gender', 'last_prod_enq2', 'first_prod_enq2']
2026-09-01 17:06:29,364 [INFO] src.common.treat_missing: TreeModelPreprocessor fit complete.
2026-09-01 17:06:29,473 [INFO] src.common.treat_missing: Random-forest data processing complete (no validation split; use oob_score=True for held-out estimates).


## 3. Random Forest Regression

In [7]:
rf_regression = Pipeline(
    steps=[
        ("model", RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features=1.0,
            random_state=42,
            n_jobs=-1
        ))
    ]
)


rf_regression.fit(
    X_train_processed,
    y_train
)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](85,)","['prospectid','total_tl','tot_closed_tl',...,'gl_flag','last_prod_enq2', 'first_prod_enq2']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,85
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2


## 4. Evaluate Random Forest Regression

In [10]:
regression_results = evaluate_regression(
    rf_regression,
    X_test_processed,
    y_test
)

print("\n==============================")
print("Random Forest Regression")
print("==============================")

for metric, value in regression_results.items():
    print(f"{metric}: {value}")


Random Forest Regression
r2: 0.8783687300983747
mae: 5.896375192277167
rmse: 7.1779039597415215


## 5. Random Search for Best Parameters

In [ ]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__max_depth": [5, 8, 10, 12, 15, 20, None],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__min_samples_leaf": [1, 2, 4, 6, 8],
    "model__max_features": [0.3, 0.5, 0.7, 1.0, "sqrt", "log2"],
    "model__ccp_alpha": [0.0, 0.001, 0.01, 0.05],
}

rf_search_base = Pipeline(
    steps=[
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
    ]
)

rf_random_search = RandomizedSearchCV(
    estimator=rf_search_base,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_random_search.fit(X_train_processed, y_train)

print("Best Parameters:")
for param, value in rf_random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (neg RMSE): {rf_random_search.best_score_:.4f}")


## 6. Run Regression Using Best Parameters

In [ ]:
# Strip the pipeline step prefix ("model__") so the params can be passed
# straight into a fresh RandomForestRegressor
best_params = {
    k.replace("model__", ""): v for k, v in rf_random_search.best_params_.items()
}

rf_best = Pipeline(
    steps=[
        ("model", RandomForestRegressor(
            **best_params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

rf_best.fit(X_train_processed, y_train)


## 7. Plot train and test loss - Model with best parameters

In [ ]:
# RandomForestRegressor has no staged/iterative loss the way boosting models do,
# so we track train/test error as trees are added using `warm_start`.

warm_start_params = {k: v for k, v in best_params.items() if k != "n_estimators"}
target_n_estimators = best_params.get("n_estimators", 200)

step = max(target_n_estimators // 20, 1)
n_estimators_range = list(range(step, target_n_estimators + step, step))

rf_warm = RandomForestRegressor(
    **warm_start_params,
    warm_start=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

train_losses = []
test_losses = []

for n_est in n_estimators_range:
    rf_warm.set_params(n_estimators=n_est)
    rf_warm.fit(X_train_processed, y_train)
    train_pred = rf_warm.predict(X_train_processed)
    test_pred = rf_warm.predict(X_test_processed)
    train_losses.append(root_mean_squared_error(y_train, train_pred))
    test_losses.append(root_mean_squared_error(y_test, test_pred))

plt.figure(figsize=(8, 5))
plt.plot(n_estimators_range, train_losses, label="Train Loss (RMSE)", marker="o")
plt.plot(n_estimators_range, test_losses, label="Test Loss (RMSE)", marker="o")
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("RMSE")
plt.title("Train vs Test Loss - Random Forest (Best Parameters)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Evaluate Random Forest Regression - Model with best parameters

In [ ]:
regression_results_best = evaluate_regression(
    rf_best,
    X_test_processed,
    y_test
)

print("\n==============================")
print("Random Forest Regression - Best Parameters")
print("==============================")

for metric, value in regression_results_best.items():
    print(f"{metric}: {value}")


## 9. Feature Importance

In [ ]:
# Make sure we have proper column names to label the importances with
if hasattr(X_train_processed, "columns"):
    feature_names = list(X_train_processed.columns)
else:
    feature_names = [f"feature_{i}" for i in range(X_train_processed.shape[1])]

# Built-in impurity-based importances
importances = rf_best.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features by impurity-based importance:")
print(feature_importance_df.head(20))

top_n = min(20, len(feature_importance_df))
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(8, 8))
plt.barh(top_features["feature"][::-1], top_features["importance"][::-1])
plt.xlabel("Feature Importance")
plt.title(f"Top {top_n} Feature Importances - Random Forest (Best Parameters)")
plt.tight_layout()
plt.show()

# Permutation importance (more reliable than impurity-based importance,
# especially with correlated/one-hot encoded features)
perm_result = permutation_importance(
    rf_best,
    X_test_processed,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 features by permutation importance:")
print(perm_importance_df.head(20))

plt.figure(figsize=(8, 8))
top_perm = perm_importance_df.head(top_n)
plt.barh(top_perm["feature"][::-1], top_perm["importance_mean"][::-1],
         xerr=top_perm["importance_std"][::-1])
plt.xlabel("Permutation Importance (mean decrease in score)")
plt.title(f"Top {top_n} Permutation Importances - Random Forest (Best Parameters)")
plt.tight_layout()
plt.show()

# Keep the features that matter - here, anything with above-average
# permutation importance. Adjust the threshold/count to taste.
important_features = perm_importance_df.loc[
    perm_importance_df["importance_mean"] > perm_importance_df["importance_mean"].mean(),
    "feature"
].tolist()

print(f"\nSelected {len(important_features)} important features out of {len(feature_names)}:")
print(important_features)


## 10. Evaluate Regression with important features and best parameters

In [ ]:
# Make sure we're working with DataFrames so we can subset by column name,
# even if the preprocessing step returned a plain numpy array
X_train_df = (
    X_train_processed if isinstance(X_train_processed, pd.DataFrame)
    else pd.DataFrame(X_train_processed, columns=feature_names)
)
X_test_df = (
    X_test_processed if isinstance(X_test_processed, pd.DataFrame)
    else pd.DataFrame(X_test_processed, columns=feature_names)
)

X_train_important = X_train_df[important_features]
X_test_important = X_test_df[important_features]

rf_best_important = Pipeline(
    steps=[
        ("model", RandomForestRegressor(
            **best_params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

rf_best_important.fit(X_train_important, y_train)

regression_results_important = evaluate_regression(
    rf_best_important,
    X_test_important,
    y_test
)

print("\n==============================")
print("Random Forest Regression - Best Parameters + Important Features")
print("==============================")

for metric, value in regression_results_important.items():
    print(f"{metric}: {value}")
